# Pregunta 3 — Quins gèneres recapten més i quins tenen millor nota?
**Projecte:** Anàlisi d'un conjunt de dades (0378 – Seguretat i alta disponibilitat, Tema 1)
**Dataset:** `IMDB-Movie-Data.csv` — 1.000 pel·lícules (2006–2016)

Aquest notebook treballa **una única pregunta de recerca**, amb el procediment complet: preparació de les dades, mètode d'anàlisi, resultat i interpretació.

## 1. Preparació de les dades

**Procediment:**
- Carreguem el CSV en un `DataFrame` de Pandas.
- Renombrem columnes amb espais/parèntesis (`runtime`, `revenue`).
- Convertim `Revenue = 0` a `NaN` (no és una recaptació real, és un valor desconegut).
- Separem `genre` en una llista de gèneres (una pel·lícula en pot tenir fins a 3).
- Creem un subconjunt `df_fiable` amb `votes >= 1000`, perquè les pel·lícules amb pocs vots tenen una nota poc representativa (sobretot les de 2016).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("IMDB-Movie-Data.csv", encoding="utf-8")
df = df.rename(columns={"Runtime (Minutes)": "runtime", "Revenue (Millions)": "revenue"})
df.columns = [c.lower() for c in df.columns]

# Revenue = 0 és, amb tota probabilitat, un valor desconegut, no una recaptació real
df.loc[df["revenue"] == 0, "revenue"] = np.nan

# Gèneres com a llista (eren text separat per comes)
df["genres"] = df["genre"].str.split(",").apply(lambda l: [g.strip() for g in l])

# Subconjunt amb vots suficients per considerar la nota fiable
df_fiable = df[df["votes"] >= 1000].copy()

print(f"Total de pel·lícules: {len(df)}")
print(f"Amb >= 1000 vots (notes fiables): {len(df_fiable)}")
df.head(3)

## 2. Pregunta de recerca

**Quins gèneres recapten més de mitjana i quins tenen millor nota mitjana?**

**Mètode:** cada pel·lícula pot tenir diversos gèneres, així que fem un `explode()` de la llista de gèneres (una fila per cada combinació pel·lícula–gènere) i calculem la mitjana de `revenue` i `rating` per gènere. Fem servir `df_fiable` (≥1000 vots) per no distorsionar la nota amb pel·lícules molt poc vistes, i ens quedem només amb gèneres que apareixen com a mínim 20 vegades perquè la mitjana sigui representativa.

In [ ]:
g = df_fiable.explode("genres")
resum_genere = (g.groupby("genres")
                  .agg(n=("title", "count"),
                       rating_mitja=("rating", "mean"),
                       revenue_mitja=("revenue", "mean"))
                  .query("n >= 20")
                  .round(2))
resum_genere.sort_values("revenue_mitja", ascending=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
resum_genere.sort_values("revenue_mitja")["revenue_mitja"].plot(kind="barh", ax=ax[0], color="steelblue")
ax[0].set_title("Recaptació mitjana per gènere (M)")
resum_genere.sort_values("rating_mitja")["rating_mitja"].plot(kind="barh", ax=ax[1], color="darkorange")
ax[1].set_title("Rating mitjà per gènere"); ax[1].set_xlim(5.5, 7.6)
plt.tight_layout(); plt.show()

## 3. Resultats

- **Animació** (191 M), **Aventura** (154 M) i **Ciència-ficció** (138 M) són els gèneres amb més recaptació mitjana — típicament pel·lícules de gran pressupost i abast familiar o de franquícia.
- **Terror** (37 M) i **Romance** (43 M) són els que menys recapten de mitjana, tot i ser gèneres habituals.
- En nota, el rànquing canvia força: **Biografia** (7,36) i **Drama** (6,99) lideren, mentre que **Animació** (7,32) combina bé nota alta i bona recaptació. **Terror** també és el gènere amb pitjor nota mitjana (6,19).
- Els gèneres que més recapten no són sempre els més ben valorats: **Acció** i **Fantasia** recapten molt però tenen una nota només mitjana (al voltant de 6,6).


## 4. Interpretació i limitacions

✏️ **Omple-ho tu:** reflexiona sobre per què Animació combina bé diners i nota (públic familiar molt fidel?). Recorda que una pel·lícula amb diversos gèneres compta en cadascun, així que els grups no són independents.